# CS 521 HW3 Problem 2
NetID: ws39

Phi-2 extracts `[left, right] operator`, SynCode checks the format, and Python does the arithmetic. `run_syncode(textual_query)` returns `(operands, operator, answer)`.

In Colab, select a GPU and run the cells in order. The first query downloads Phi-2 and builds the grammar cache.


In [ ]:
%pip -q install "syncode==0.4.16" "transformers==4.53.2" "accelerate==1.10.1"


## Grammar and examples
The four examples show how to extract each operation, without giving the answers. Order matters: “subtract A from B” means B - A.


In [ ]:
"""CS 521 HW3 Problem 2: extract with Phi-2 and SynCode, then calculate in Python."""
import math
import operator
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria
from syncode import Grammar, SyncodeLogitsProcessor

MODEL_ID = "microsoft/phi-2"
NUMBER = r"[+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?"
GRAMMAR = r"""
start: number "," number "]" operation
number: sign? mantissa exponent?
sign: "+" | "-"
mantissa: DIGITS ("." DIGITS?)? | "." DIGITS
exponent: ("e" | "E") sign? DIGITS
DIGITS: /[0-9]+/
operation: "+" | "-" | "*" | "/"
%ignore /[ \t]+/
"""
RECORD = re.compile(rf"\s*\[\s*({NUMBER})\s*,\s*({NUMBER})\s*\]\s*([+*/-])\s*")
OPERATIONS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}

PROMPT = """Extract two operands and one arithmetic operator from the question.
Write only [left_operand, right_operand] operator. Do not calculate the answer.
Use the order needed for left_operand operator right_operand.
For "subtract A from B", the order is [B, A]. For "A divided by B", it is [A, B].
Copy the complete numeric literals exactly: keep every minus sign, decimal point, and exponent.
A negative operand stays negative; scientific notation stays in scientific notation.

Question: What is 14.25 plus 6.5?
Extraction: [ 14.25, 6.5] +

Question: Subtract 3.75 from 18.5.
Extraction: [ 18.5, 3.75] -

Question: What is -7.25 multiplied by 3.0?
Extraction: [ -7.25, 3.0] *

Question: What is 1.2e3 divided by 6.0?
Extraction: [ 1.2e3, 6.0] /
"""



## Model and calculator
The prompt ends with `[`, and SynCode constrains the rest of the record. I reset the decoder for each query and stop once the record is complete. The model is loaded only once.


In [ ]:
_model = _tokenizer = _processor = None


def load_model():
    global _model, _tokenizer, _processor
    if _model is not None:
        return
    if torch.cuda.is_available():
        device, dtype = "cuda", torch.float32
    elif torch.backends.mps.is_available():
        device, dtype = "mps", torch.bfloat16
    else:
        device, dtype = "cpu", torch.float32
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True,
        attn_implementation="sdpa", device_map={"": device},
    ).eval()
    _processor = SyncodeLogitsProcessor(
        grammar=Grammar(GRAMMAR), tokenizer=_tokenizer,
        parse_output_only=True, mode="grammar_mask",
    )


class CompleteRecord(StoppingCriteria):
    def __init__(self, prompt_length):
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        text = "[" + _tokenizer.decode(input_ids[0, self.prompt_length:], skip_special_tokens=True)
        return RECORD.fullmatch(text) is not None


def extract(textual_query):
    if not isinstance(textual_query, str) or not textual_query.strip():
        raise ValueError("The query must be a nonempty string.")
    load_model()
    prompt = PROMPT + "\nQuestion: " + textual_query.strip() + "\nExtraction: [ "
    inputs = _tokenizer(prompt, return_tensors="pt").to(_model.device)
    prompt_length = inputs.input_ids.shape[1]
    if prompt_length + 96 > _model.config.max_position_embeddings:
        raise ValueError("The query is too long for Phi-2.")
    _processor.reset()
    with torch.inference_mode():
        output = _model.generate(
            **inputs, max_new_tokens=96, do_sample=False,
            pad_token_id=_tokenizer.eos_token_id,
            logits_processor=[_processor],
            stopping_criteria=[CompleteRecord(prompt_length)],
        )
    text = "[" + _tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True)
    match = RECORD.fullmatch(text)
    if match is None:
        raise ValueError(f"Incomplete extraction: {text!r}")
    operands = [float(match[1]), float(match[2])]
    if not all(math.isfinite(x) for x in operands):
        raise ValueError("Operands must be finite.")
    return operands, match[3], text


def run_syncode(textual_query):
    operands, op, _ = extract(textual_query)
    answer = float(OPERATIONS[op](*operands))
    return operands, op, answer


## Tests
The first three tests come from the assignment. I also check operand order, negative numbers, decimals, exponent notation, zero, and repeated calls.


In [ ]:
cases = [('What is 327. multiplied by 11.0?', [327.0, 11.0], '*', 3597.0), ('What is 45.1 plus 23.54?', [45.1, 23.54], '+', 68.64), ('What is 120.4 divided by 4.0?', [120.4, 4.0], '/', 30.1), ('What is 17.25 minus 3.5?', [17.25, 3.5], '-', 13.75), ('Subtract 7.25 from 20.5.', [20.5, 7.25], '-', 13.25), ('What is -3.5 multiplied by 2.0?', [-3.5, 2.0], '*', -7.0), ('What is .75 plus 0.125?', [0.75, 0.125], '+', 0.875), ('What is 8.0 divided by -2.0?', [8.0, -2.0], '/', -4.0), ('What is 5.0 minus -2.5?', [5.0, -2.5], '-', 7.5), ('What is 0.0 multiplied by 18.25?', [0.0, 18.25], '*', 0.0), ('What is 2.5e2 divided by 5.0?', [250.0, 5.0], '/', 50.0), ('Divide 9.0 by 4.0.', [9.0, 4.0], '/', 2.25), ('What is 1000.25 plus 0.75?', [1000.25, 0.75], '+', 1001.0), ('What is 0.1 plus 0.2?', [0.1, 0.2], '+', 0.3), ('What is -4.25 plus 1.5?', [-4.25, 1.5], '+', -2.75), ('What is -6.0 minus -2.0?', [-6.0, -2.0], '-', -4.0), ('What is -3.0 multiplied by -4.0?', [-3.0, -4.0], '*', 12.0), ('What is 9.5 divided by 0.5?', [9.5, 0.5], '/', 19.0), ('Subtract 19.25 from 4.0.', [4.0, 19.25], '-', -15.25), ('What is 0.25 times 40.0?', [0.25, 40.0], '*', 10.0), ('What is the product of 8.25 and 2.0?', [8.25, 2.0], '*', 16.5), ('What is the sum of 3.125 and 6.875?', [3.125, 6.875], '+', 10.0), ('Divide .5 by .25.', [0.5, 0.25], '/', 2.0), ('What is 31.5 divided by 7.0?', [31.5, 7.0], '/', 4.5), ('What is 4e-2 plus 0.06?', [0.04, 0.06], '+', 0.1), ('What is 10.0 minus 15.0?', [10.0, 15.0], '-', -5.0), ('What is 126.75 multiplied by 12.4?', [126.75, 12.4], '*', 1571.7), ('What is 327. multiplied by 11.0?', [327.0, 11.0], '*', 3597.0)]

passed = 0
for query, expected_operands, expected_op, expected_answer in cases:
    actual = run_syncode(query)
    ok = (isinstance(actual, tuple) and len(actual) == 3
          and isinstance(actual[0], list) and all(type(x) is float for x in actual[0])
          and actual[0] == expected_operands and actual[1] == expected_op
          and type(actual[2]) is float
          and math.isclose(actual[2], expected_answer, rel_tol=1e-9, abs_tol=1e-9))
    passed += int(ok)
    print(query, actual, "PASS" if ok else "FAIL")
print(f"Passed {passed}/{len(cases)}")


## Local results

All 28 local tests passed with Phi-2 on Apple M3 (MPS, bfloat16), SynCode 0.4.16, and Transformers 4.53.2. The three assignment examples are shown below. The Colab code uses float32 on CUDA; I have not tested that runtime.

| Query | Returned tuple | Passed |
|---|---|---|
| What is 327. multiplied by 11.0? | `([327.0, 11.0], '*', 3597.0)` | True |
| What is 45.1 plus 23.54? | `([45.1, 23.54], '+', 68.64)` | True |
| What is 120.4 divided by 4.0? | `([120.4, 4.0], '/', 30.1)` | True |


## References
- [SynCode logits processor example](https://github.com/structuredllm/syncode/blob/main/notebooks/example_logits_processor.ipynb)
- [Microsoft Phi-2](https://huggingface.co/microsoft/phi-2)
